In [ ]:
# !pip install pycryptodome # uncomment this line if you haven't installed the library or if you are using online environments like Google Colab

In [ ]:
tmp = b'Sixteen byte key'
print(tmp)

In [ ]:
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
from Crypto.Random import get_random_bytes
import binascii

# 1. Κλειδί 16 bytes (128 bits) για τον AES
key = b'Sixteen byte key'
# 2. Τα δεδομένα μας
# 32 bytes (2 blocks) όλα ίδια
plaintext = b'Hello crypto. This is a test msg' # 32
print(f"Original Plaintext: {plaintext}\n")


In [ ]:
# --- ΔΟΚΙΜΗ 1: ECB ---
print("--- AES σε ECB Mode ---")
cipher_ecb = AES.new(key, AES.MODE_ECB)
# Κρυπτογραφούμε (εδώ είναι ακριβώς 32 bytes, άρα δεν χρειάζεται padding)
ciphertext_ecb = cipher_ecb.encrypt(plaintext)
hex_ecb = binascii.hexlify(ciphertext_ecb).decode('utf-8')
print(f"Ciphertext (Hex): {hex_ecb}")
print(len(hex_ecb))
print(f"Block 1: {hex_ecb[:32]}")
print(f"Block 2: {hex_ecb[32:]}\n")

In [ ]:
# --- ΔΟΚΙΜΗ 2: CBC MODE ---
print("--- AES σε CBC Mode ---")
# Δημιουργούμε ένα τυχαίο Initialization Vector (16 bytes)
iv = get_random_bytes(16)
cipher_cbc = AES.new(key, AES.MODE_CBC, iv)

ciphertext_cbc = cipher_cbc.encrypt(plaintext)
hex_cbc = binascii.hexlify(ciphertext_cbc).decode('utf-8')

print(f"IV (Hex): {binascii.hexlify(iv).decode('utf-8')}")
print(f"Ciphertext (Hex): {hex_cbc}")
print(f"Block 1: {hex_cbc[:32]}")
print(f"Block 2: {hex_cbc[32:]}\n")

In [ ]:
# --- ΑΠΟΚΡΥΠΤΟΓΡΑΦΗΣΗ CBC ΜΕ PADDING ---
# Ας δοκιμάσουμε τώρα με κείμενο που δεν είναι ακριβώς πολλαπλάσιο του 16
message = b"This is a secret message that needs paddin!"
print(len(message))
print("--- Αποκρυπτογράφηση & Padding ---")

# Padding
padded_message = pad(message, AES.block_size)
print(f"Padded Message: {padded_message}")
print(len(padded_message))

# Κρυπτογράφηση
cipher_encrypt = AES.new(key, AES.MODE_CBC, iv)
ciphered_data = cipher_encrypt.encrypt(padded_message)
print(f"Encrypted Message: {binascii.hexlify(ciphered_data).decode('utf-8')}")

# Αποκρυπτογράφηση (Χρειαζόμαστε νέο αντικείμενο cipher με το ίδιο IV και κλειδί)
cipher_decrypt = AES.new(key, AES.MODE_CBC, iv)
decrypted_padded = cipher_decrypt.decrypt(ciphered_data)

# Unpadding (Αφαίρεση των extra bytes)
decrypted_message = unpad(decrypted_padded, AES.block_size)
print(f"Decrypted Message: {decrypted_message.decode('utf-8')}")


In [ ]:
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad
import os

def decrypt_bmp_ecb(input_filename, output_filename, key):
    # 1. Διαβάζουμε το αρχείο σε δυαδική μορφή (binary)
    try:
        with open(input_filename, 'rb') as f:
            image_data = f.read()
    except FileNotFoundError:
        print("Το αρχείο δεν βρέθηκε! Σιγουρευτείτε ότι έχετε ένα .bmp αρχείο στον φάκελο.")
        return

    header_size = 54
    header = image_data[:header_size]
    body = image_data[header_size:]

    padded_body = pad(body, AES.block_size)

    cipher_ecb = AES.new(key, AES.MODE_CBC, iv)
    decrypted_body = cipher_ecb.decrypt(padded_body)

    # 5. Ενώνουμε το αρχικό (ακρυπτογράφητο) header με το κρυπτογραφημένο σώμα
    decrypted_image_data = header + decrypted_body

    # 6. Αποθηκεύουμε το νέο αρχείο
    with open(output_filename, 'wb') as f:
        f.write(decrypted_image_data)

    print(f"Η εικόνα αποθηκεύτηκε επιτυχώς ως {output_filename}")


def encrypt_bmp_ecb(input_filename, output_filename, key):
    # 1. Διαβάζουμε το αρχείο σε δυαδική μορφή (binary)
    try:
        with open(input_filename, 'rb') as f:
            image_data = f.read()
    except FileNotFoundError:
        print("Το αρχείο δεν βρέθηκε! Σιγουρευτείτε ότι έχετε ένα .bmp αρχείο στον φάκελο.")
        return

    # 2. Διαχωρίζουμε το Header από τα δεδομένα των Pixels
    # Σε ένα τυπικό 24-bit BMP, το header είναι 54 bytes.
    header_size = 54
    header = image_data[:header_size]
    body = image_data[header_size:]

    # 3. Εφαρμόζουμε Padding στα δεδομένα της εικόνας
    # Ο AES χρειάζεται μπλοκ των 16 bytes.
    padded_body = pad(body, AES.block_size)

    # 4. Κρυπτογράφηση σε ECB Mode
    cipher_ecb = AES.new(key, AES.MODE_CBC, iv)
    encrypted_body = cipher_ecb.encrypt(padded_body)

    # 5. Ενώνουμε το αρχικό (ακρυπτογράφητο) header με το κρυπτογραφημένο σώμα
    encrypted_image_data = header + encrypted_body

    # 6. Αποθηκεύουμε το νέο αρχείο
    with open(output_filename, 'wb') as f:
        f.write(encrypted_image_data)

    print(f"Η εικόνα αποθηκεύτηκε επιτυχώς ως {output_filename}")


# --- Εκτέλεση του Προγράμματος ---

# Δημιουργούμε ένα τυχαίο κλειδί 16 bytes (128 bits) για τον AES
my_key = os.urandom(16)
print(my_key)

# Ονόματα αρχείων (Βεβαιωθείτε ότι το 'tux.bmp' υπάρχει)
input_file = 'panda.bmp'
output_ecb_file = 'panda_encrypted_ecb.bmp'

encrypt_bmp_ecb(input_file, output_ecb_file, my_key)

decrypt_bmp_ecb(output_ecb_file, "decrypted.bmp", my_key)